# Analiza portfela XTB

Wczytaj eksporty CSV do `data/`, uruchom wszystkie komórki. Analiza obejmuje:
- wartość portfela z widocznymi skokami wpłat,
- stopy zwrotu trzema metodami: **ROI**, **MWR/XIRR** i **TWR**,
- porównanie zwrotu z S&P 500 TR i NASDAQ 100 (te same wpłaty w te same dni),
- **atrybucję walutową** – ile wyniku dały inwestycje, a ile kursy walut,
- metryki ryzyka na bazie TWR (zmienność, Sharpe, drawdown, VaR, heatmapa miesięczna),
- koszt przewalutowań, strukturę i sektory portfela.

Pełny interaktywny **raport HTML** generuje ostatnia komórka (albo `python -m xtb_analyzer` z katalogu projektu).

> **Setup:** zalecane `pip install -e ".[dev]"` z katalogu głównego projektu i restart kernela. Notebook dodaje też `src/` do ścieżki importu.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from IPython.display import display

from xtb_analyzer.attribution import compute_fx_attribution
from xtb_analyzer.benchmark import build_benchmark_suite
from xtb_analyzer.charts import (
    PLOTLY_SHOW_CONFIG,
    prepare_return_comparison_data,
    show_return_comparison_widget,
    plot_conversion_losses,
    plot_currency_allocation,
    plot_drawdown,
    plot_fx_attribution_series,
    plot_fx_attribution_waterfall,
    plot_holdings_allocation,
    plot_monthly_returns_heatmap,
    plot_nav_with_deposits,
    plot_return_dashboard,
    plot_return_measures,
    plot_rolling_volatility,
    plot_sector_allocation,
)
from xtb_analyzer.flows import analyze_conversion_losses, attach_deposits_to_nav, conversion_summary
from xtb_analyzer.metrics import (
    build_report,
    compute_metrics,
    format_metrics_table,
    format_realized_pnl_table,
    realized_pnl_summary,
)
from xtb_analyzer.parser import load_accounts
from xtb_analyzer.portfolio import build_nav_series
from xtb_analyzer.report import generate_report
from xtb_analyzer.returns import compute_return_measures, twr_daily_returns
from xtb_analyzer.risk import compute_risk_stats
from xtb_analyzer.sectors import enrich_positions_with_sectors, sector_allocation

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../output")
REPORT_CURRENCY = "pln"
REFRESH_MARKET_DATA = True

CSV_FILES = sorted(DATA_DIR.glob("XTB_*_account_cs.csv"))
if not CSV_FILES:
    raise FileNotFoundError("Brak plikow CSV w data/.")

accounts = load_accounts(CSV_FILES)
nav, holdings_ts, positions = build_nav_series(accounts, live_prices=REFRESH_MARKET_DATA)
nav = attach_deposits_to_nav(nav, accounts)
positions = enrich_positions_with_sectors(positions)
conversion_losses = analyze_conversion_losses(accounts)
conversion_stats = conversion_summary(conversion_losses)
benchmarks = build_benchmark_suite(
    accounts,
    nav.index,
    currency=REPORT_CURRENCY,
    refresh_latest=REFRESH_MARKET_DATA,
)
sp500_benchmark = benchmarks["sp500"]
nasdaq100_benchmark = benchmarks["nasdaq100"]
acwi_benchmark = benchmarks["acwi"]
return_comparison_data = prepare_return_comparison_data(nav, benchmarks, REPORT_CURRENCY)

print("Konta:", [a.account_id for a in accounts])
print("Okres:", nav.index[0].date(), "->", nav.index[-1].date())


Konta: ['55287715', '53287226']
Okres: 2025-11-04 -> 2026-08-05


## Aktualna stopa zwrotu

In [2]:
plot_return_dashboard(nav, REPORT_CURRENCY).show(config=PLOTLY_SHOW_CONFIG)

## Wartość portfela w czasie (ze skokami wpłat)

In [3]:
plot_nav_with_deposits(nav, REPORT_CURRENCY).show(config=PLOTLY_SHOW_CONFIG)

## Stopy zwrotu: ROI vs MWR (XIRR) vs TWR

- **ROI** – wartość ÷ wpłaty − 1; nie uwzględnia, kiedy pieniądze trafiły do portfela.
- **MWR/XIRR** – roczna stopa ważona pieniędzmi; faktyczne doświadczenie inwestora z uwzględnieniem momentów wpłat.
- **TWR** – stopa ważona czasem; neutralizuje wpłaty i mierzy jakość decyzji inwestycyjnych (porównywalna z indeksami).

In [4]:
measures = compute_return_measures(accounts, nav, REPORT_CURRENCY)
twr_returns = twr_daily_returns(nav, REPORT_CURRENCY)

print(f"ROI (prosty zwrot):   {measures.roi:+.2%}")
if measures.mwr is not None:
    print(f"MWR (za okres):       {measures.mwr:+.2%}   (XIRR w skali roku: {measures.xirr:+.2%})")
print(f"TWR (za okres):       {measures.twr:+.2%}", end="")
print(f"   (w skali roku: {measures.twr_annualized:+.2%})" if measures.twr_annualized is not None else "")

plot_return_measures(measures).show(config=PLOTLY_SHOW_CONFIG)

ROI (prosty zwrot):     +30.29%
MWR / XIRR (roczna):    +52.43%
TWR (skumulowana):      +31.24%
TWR (roczna):           +43.68%


## Zwrot z kapitału vs benchmarki (dane miesięczne)


In [5]:
show_return_comparison_widget(
    nav,
    benchmarks,
    REPORT_CURRENCY,
    data=return_comparison_data,
)


## Atrybucja walutowa: inwestycje vs waluta

Rozbicie łącznego wyniku w PLN na część wypracowaną przez inwestycje oraz część wynikającą ze zmian kursów walut (w tym koszt przewalutowania XTB względem kursu rynkowego).

In [6]:
attribution = compute_fx_attribution(accounts, nav)

print(f"Zysk z inwestycji: {attribution.investment_gain_pln:+,.2f} PLN")
print(f"Zysk walutowy:     {attribution.fx_gain_pln:+,.2f} PLN")
print(f"Wynik łączny:      {attribution.total_gain_pln:+,.2f} PLN")

plot_fx_attribution_waterfall(attribution).show(config=PLOTLY_SHOW_CONFIG)
plot_fx_attribution_series(attribution).show(config=PLOTLY_SHOW_CONFIG)
display(attribution.by_currency.round(2))

Zysk z inwestycji: +3,215.55 PLN
Zysk walutowy:     +67.97 PLN
Wynik łączny:      +3,283.53 PLN


,currency,pln_spent,units_received,avg_deposit_rate,current_rate,current_value_local,investment_gain_pln,fx_gain_pln,total_gain_pln
0,USD,10089.72,2725.85,3.70,3.72,3636.23,3389.88,60.26,3450.14
1,EUR,749.97,176.17,4.26,4.30,135.64,-174.33,7.72,-166.61


## Ryzyko (na bazie dziennych stóp TWR)

In [7]:
risk = compute_risk_stats(twr_returns)

risk_table = pd.DataFrame(
    [
        ("Zmienność (roczna)", f"{risk.volatility_annual:.2%}"),
        ("Współczynnik Sharpe'a", f"{risk.sharpe:.2f}"),
        ("Współczynnik Sortino", f"{risk.sortino:.2f}"),
        ("Maks. obsunięcie", f"{risk.max_drawdown:.2%}"),
        ("Okres maks. obsunięcia", f"{risk.max_drawdown_start} → {risk.max_drawdown_end}"),
        ("Calmar", f"{risk.calmar:.2f}" if risk.calmar is not None else "—"),
        ("VaR 95% (dzienny)", f"{risk.var_95_daily:.2%}"),
        ("Najlepszy / najgorszy dzień", f"{risk.best_day:+.2%} / {risk.worst_day:+.2%}"),
        (
            "Najlepszy / najgorszy miesiąc",
            f"{risk.best_month:+.2%} / {risk.worst_month:+.2%}"
            if risk.best_month is not None
            else "—",
        ),
        (
            "Miesiące na plusie",
            f"{risk.positive_months_pct:.0%}" if risk.positive_months_pct is not None else "—",
        ),
    ],
    columns=["Metryka", "Wartość"],
)
display(risk_table)

plot_drawdown(twr_returns, REPORT_CURRENCY).show(config=PLOTLY_SHOW_CONFIG)
plot_monthly_returns_heatmap(twr_returns, REPORT_CURRENCY).show(config=PLOTLY_SHOW_CONFIG)
plot_rolling_volatility(twr_returns, currency=REPORT_CURRENCY).show(config=PLOTLY_SHOW_CONFIG)

,Metryka,Wartość
0,Zmienność (roczna),36.31%
1,Współczynnik Sharpe'a,1.15
2,Współczynnik Sortino,1.78
3,Maks. obsunięcie,-23.14%
4,Okres maks. obsunięcia,2025-11-05 → 2026-02-23
5,Calmar,1.89
6,VaR 95% (dzienny),-3.05%
7,Najlepszy / najgorszy dzień,+11.59% / -10.18%
8,Najlepszy / najgorszy miesiąc,+36.04% / -10.68%
9,Miesiące na plusie,60%


## Koszt przewalutowań (prowizja XTB 0,5%)

In [8]:
plot_conversion_losses(conversion_losses, conversion_stats).show(config=PLOTLY_SHOW_CONFIG)

## Pozycje, sektory i metryki ryzyka

In [9]:
plot_holdings_allocation(positions).show(config=PLOTLY_SHOW_CONFIG)
plot_currency_allocation(positions).show(config=PLOTLY_SHOW_CONFIG)
plot_sector_allocation(sector_allocation(positions)).show(config=PLOTLY_SHOW_CONFIG)

display(positions.sort_values("market_value_pln", ascending=False))
display(format_metrics_table(compute_metrics(nav, REPORT_CURRENCY), currency=REPORT_CURRENCY))
display(format_realized_pnl_table(realized_pnl_summary(accounts)))

,account_id,symbol,currency,shares,avg_cost,market_price,cost_basis,market_value,unrealized_pnl,market_value_pln,market_value_usd,market_value_eur,weight,sector,industry
4,53287226,NVDA.US,USD,5.6500,210.704071,221.009995,1190.478000,1248.706469,58.228469,4649.683296,1248.706469,1080.880272,0.329361,Technology,Semiconductors
0,53287226,UBER.US,USD,13.0000,84.765286,67.959999,1101.948712,883.479988,-218.468724,3289.726005,883.479988,764.740244,0.233028,Technology,Software - Application
2,53287226,NOW.US,USD,5.0000,103.920124,116.824997,519.600622,584.124985,64.524363,2175.047741,584.124985,505.618565,0.154070,Technology,Software - Application
1,53287226,GOOGL.US,USD,1.0000,366.640445,362.220001,366.640445,362.220001,-4.420444,1348.762364,362.220001,313.537619,0.095540,Communication Services,Internet Content & Information
8,53287226,CRDO.US,USD,1.0000,221.920000,228.830002,221.920000,228.830002,6.910002,852.071374,228.830002,198.075241,0.060357,Technology,Semiconductors
5,53287226,NU.US,USD,12.6000,12.923492,14.375000,162.836000,181.125000,18.289000,674.437034,181.125000,156.781793,0.047774,Financial Services,Banks - Regional
3,55287715,IFX.DE,EUR,2.2546,78.058871,60.080002,175.991530,135.456372,-40.535158,582.584309,156.488422,135.456372,0.041267,Technology,Semiconductors
7,53287226,ACMR.US,USD,1.3006,83.002357,81.010002,107.952866,105.361609,-2.591257,392.324477,105.361609,91.201005,0.027790,Technology,Semiconductor Equipment & Materials
6,53287226,POET.US,USD,5.0000,12.790000,8.200000,63.950000,40.999999,-22.950001,152.667593,40.999999,35.489598,0.010814,Technology,Semiconductors


,Metryka,Wartość
0,Aktualna wartość portfela,"14 123,22 PLN"
1,Suma wpłat,"10 839,69 PLN"
2,Zysk / strata,"3 283,53 PLN"
3,Zwrot na wpłaconym kapitale,"+30,29% (wartość ÷ wpłaty − 1)"
4,Zmienność (roczna),"+987,93%"
5,Współczynnik Sharpe'a,"2,63"
6,Współczynnik Sortino,"10,10"
7,Maks. obsunięcie (drawdown),"-20,36%"
8,Szczyt wartości portfela (ATH),"14 580,22 PLN"
9,Odległość od ATH,"-3,13%"


,Symbol,Transakcje,Zrealizowany P&L,Średni P&L,Win rate
0,AMD.US,10,1012.08,101.21,100.0
1,AMZN.US,4,-2.99,-0.75,25.0


## Eksport

In [10]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
build_report(accounts, nav, positions, output_dir=OUTPUT_DIR)
nav.to_csv(OUTPUT_DIR / "nav_timeseries.csv")
conversion_losses.to_csv(OUTPUT_DIR / "conversion_losses.csv", index=False)
sp500_benchmark.to_csv(OUTPUT_DIR / "sp500_benchmark.csv")
nasdaq100_benchmark.to_csv(OUTPUT_DIR / "nasdaq100_benchmark.csv")
acwi_benchmark.to_csv(OUTPUT_DIR / "acwi_benchmark.csv")

report_path = generate_report(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    currency=REPORT_CURRENCY,
    refresh_latest=REFRESH_MARKET_DATA,
)
print(f"Zapisano do {OUTPUT_DIR.resolve()}")
print(f"Raport HTML: {report_path.resolve()}")

Zapisano do C:\Users\oliwi\OneDrive\Desktop\xtb_portfolio_analyzer\output
Raport HTML: C:\Users\oliwi\OneDrive\Desktop\xtb_portfolio_analyzer\output\report.html
